<a href="https://colab.research.google.com/github/PARODBE/Clase/blob/main/Ejercicio_Filmaffinity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Vamos a hacer un análisis del total de candidaturas al Oscar que tienen las películas más famosas por género.

Para ello vamos a extraer información de los TOP películas de filmaffinity (https://www.filmaffinity.com/es/topgen.php).
Concretamente obtén el top 30 películas de los siguientes géneros: Acción, Comedia, Drama, Terror. Además escoge películas del año 1980 en adelante.

Para ver los candidatos al óscar a mejor actor/actriz/actor de reparto/actriz de reparto podemos utilizar la wikipedia:

https://es.wikipedia.org/wiki/Anexo:Óscar_al_mejor_actor

**Existen anexos equivalentes para cada posible premio.



Una vez combinadas y transformadas ambas fuentes, contesta a las siguientes preguntas:

- Cuál es el género con mayor número de candidaturas?

- Cuál es la película con mayor diferencia porcentual de candidaturas respecto a la media de su género?

In [ ]:
%%capture
!pip install google-colab-selenium

In [ ]:
import google_colab_selenium as gs
from selenium.webdriver.chrome.options import Options

options = Options()
# AÑADIMOS USER AGENT
options.add_argument(f"--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36")
driver = gs.Chrome(options=options)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## FILMAFINITY

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


full_films = []
genres = ['accion', 'comedia', 'drama', 'terror']
for genre in genres:
      genre_url = f"https://www.filmaffinity.com/es/topgen.php?genres=%2B{genre[:2].upper()}&chv=0&orderby=rc&movietype=movie%7C&country=&fromyear=1980&toyear=2025&ratingcount=2&runtimemin=0&runtimemax=4"
      # driver.get(genre_url)
      # source = driver.page_source
      # headers = {
      #     "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36",
      #     # El user agent es una huella que deja el navegador cuando accede a las páginas web
      # }
      # res = requests.get(genre_url, headers=headers)
      # source = res.text

      with open(f"filmaffinity-{genre}.html") as f:
        source = f.read()

      soup = BeautifulSoup(source, 'lxml')

      for film in soup.select('#top-movies li')[:30]:

        title = film.select_one(".stretched-link")
        year = film.select_one('.mc-year')
        actores = [actor.text for actor in film.select('.nb') if actor]

        full_films.append({
          "title": title.text.strip() if title else None,
          "year": year.text.strip() if year else None,
          "genre": genre,
          "actors": actores
          })

full_films

[{'title': 'Kill Bill: The Whole Bloody Affair',
  'year': '2004',
  'genre': 'accion',
  'actors': ['Quentin Tarantino',
   'Uma Thurman,',
   'Lucy Liu,',
   'Vivica A. Fox']},
 {'title': 'Ciudad de Dios',
  'year': '2002',
  'genre': 'accion',
  'actors': ['Fernando Meirelles,',
   'Kátia Lund',
   'Alexandre Rodrigues,',
   'Leandro Firmino,',
   'Phellipe Haagensen']},
 {'title': 'El señor de los anillos: El retorno del rey',
  'year': '2003',
  'genre': 'accion',
  'actors': ['Peter Jackson',
   'Elijah Wood,',
   'Viggo Mortensen,',
   'Ian McKellen']},
 {'title': 'Attack on Titan: The Last Attack',
  'year': '2024',
  'genre': 'accion',
  'actors': ['Yuichiro Hayashi']},
 {'title': 'Reservoir Dogs',
  'year': '1992',
  'genre': 'accion',
  'actors': ['Quentin Tarantino',
   'Tim Roth,',
   'Harvey Keitel,',
   'Steve Buscemi']},
 {'title': None, 'year': None, 'genre': 'accion', 'actors': []},
 {'title': 'El caballero oscuro',
  'year': '2008',
  'genre': 'accion',
  'actors': [

In [ ]:
movies = pd.DataFrame(full_films)
movies

,title,year,genre,actors
0,Kill Bill: The Whole Bloody Affair,2004,accion,"[Quentin Tarantino, Uma Thurman,, Lucy Liu,, V..."
1,Ciudad de Dios,2002,accion,"[Fernando Meirelles,, Kátia Lund, Alexandre Ro..."
2,El señor de los anillos: El retorno del rey,2003,accion,"[Peter Jackson, Elijah Wood,, Viggo Mortensen,..."
3,Attack on Titan: The Last Attack,2024,accion,[Yuichiro Hayashi]
4,Reservoir Dogs,1992,accion,"[Quentin Tarantino, Tim Roth,, Harvey Keitel,,..."
...,...,...,...,...
115,El corazón del ángel,1987,terror,"[Alan Parker, Mickey Rourke,, Robert De Niro,,..."
116,Cube,1997,terror,"[Vincenzo Natali, Maurice Dean Wint,, Nicole d..."
117,La mosca,1986,terror,"[David Cronenberg, Jeff Goldblum,, Geena Davis..."
118,One Cut of The Dead,2017,terror,"[Shinichirô Ueda, Takayuki Hamatsu,, Yuzuki Ak..."


In [ ]:
movies.to_csv("filmaffinity-top30.csv")

##WIKIPEDIA

In [ ]:
import requests
import pandas as pd
from io import StringIO

actor_types = [
    ("al", "actor"),
    ("a_la", "actriz"),
    ("al", "actor_de_reparto"),
    ("a_la", "actriz_de_reparto")
]

headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/51.0.2704.103 Safari/537.36"
    # El user agent es una huella que deja el navegador cuando accede a las páginas web
}


all_tables = []
for art, actor_type in actor_types:
  url = f'https://es.wikipedia.org/wiki/Anexo:%C3%93scar_{art}_mejor_{actor_type}'
  res = requests.get(url, headers=headers)
  dfs = pd.read_html(StringIO(res.text), encoding = 'utf8')[3:13]
  all_tables.extend(dfs)

In [ ]:
[len(x.columns) for x in all_tables]

[5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 7,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5]

In [ ]:
df_to_keep = [i for i, x in enumerate(all_tables) if len(x.columns) == 5 and "Año" in x.columns]
clean_tables = []
for df in [all_tables[i] for i in df_to_keep]:
  df.columns = ["year", "na", "name", "movie", "papel"]
  df = df.drop(columns=["na", "movie", "papel"])
  clean_tables.append(df)

oscars = pd.concat(clean_tables)

In [ ]:
oscars

,year,name
0,NaN,NaN
1,2025.0,Michael B. Jordan
2,2025.0,Timothée Chalamet
3,2025.0,Leonardo DiCaprio
4,2025.0,Ethan Hawke
...,...,...
19,1936.0,Gale Sondergaard
20,1936.0,Beulah Bondi
21,1936.0,Alice Brady
22,1936.0,Bonita Granville


In [ ]:
oscars = oscars.groupby("name").size().sort_values()

In [ ]:
oscars

,0
name,
Alexander Knox,1
Alec Baldwin,1
Albert Brooks,1
Albert Basserman,1
Alan Bates,1
...,...
Laurence Olivier,10
Bette Davis,11
Jack Nicholson,12


In [ ]:
oscars = oscars.reset_index()
oscars.columns = ["name", "nominations"]

In [ ]:
oscars

,name,nominations
0,Alexander Knox,1
1,Alec Baldwin,1
2,Albert Brooks,1
3,Albert Basserman,1
4,Alan Bates,1
...,...,...
961,Laurence Olivier,10
962,Bette Davis,11
963,Jack Nicholson,12
964,Katharine Hepburn,12


#COMBINAR DATOS Y EXPLOTAR

In [ ]:
movies = movies.explode("actors")
movies

,title,year,genre,actors
0,Kill Bill: The Whole Bloody Affair,2004,accion,Quentin Tarantino
0,Kill Bill: The Whole Bloody Affair,2004,accion,"Uma Thurman,"
0,Kill Bill: The Whole Bloody Affair,2004,accion,"Lucy Liu,"
0,Kill Bill: The Whole Bloody Affair,2004,accion,Vivica A. Fox
1,Ciudad de Dios,2002,accion,"Fernando Meirelles,"
...,...,...,...,...
118,One Cut of The Dead,2017,terror,Harumi Shuhama
119,Obsession,2025,terror,Curry Barker
119,Obsession,2025,terror,"Michael Johnston,"
119,Obsession,2025,terror,"Inde Navarrette,"


In [ ]:
movies["actors"] = movies["actors"].astype(str).apply(lambda x: x[:-1] if x.endswith(",") else x)
movies

,title,year,genre,actors
0,Kill Bill: The Whole Bloody Affair,2004,accion,Quentin Tarantino
0,Kill Bill: The Whole Bloody Affair,2004,accion,Uma Thurman
0,Kill Bill: The Whole Bloody Affair,2004,accion,Lucy Liu
0,Kill Bill: The Whole Bloody Affair,2004,accion,Vivica A. Fox
1,Ciudad de Dios,2002,accion,Fernando Meirelles
...,...,...,...,...
118,One Cut of The Dead,2017,terror,Harumi Shuhama
119,Obsession,2025,terror,Curry Barker
119,Obsession,2025,terror,Michael Johnston
119,Obsession,2025,terror,Inde Navarrette


In [ ]:
movies["actors"] = movies["actors"].str.lower().str.strip()
movies

,title,year,genre,actors
0,Kill Bill: The Whole Bloody Affair,2004,accion,quentin tarantino
0,Kill Bill: The Whole Bloody Affair,2004,accion,uma thurman
0,Kill Bill: The Whole Bloody Affair,2004,accion,lucy liu
0,Kill Bill: The Whole Bloody Affair,2004,accion,vivica a. fox
1,Ciudad de Dios,2002,accion,fernando meirelles
...,...,...,...,...
118,One Cut of The Dead,2017,terror,harumi shuhama
119,Obsession,2025,terror,curry barker
119,Obsession,2025,terror,michael johnston
119,Obsession,2025,terror,inde navarrette


In [ ]:
oscars["name"] = oscars["name"].str.lower().str.strip()
oscars

,name,nominations
0,alexander knox,1
1,alec baldwin,1
2,albert brooks,1
3,albert basserman,1
4,alan bates,1
...,...,...
961,laurence olivier,10
962,bette davis,11
963,jack nicholson,12
964,katharine hepburn,12


In [ ]:
df = movies.merge(oscars, left_on="actors", right_on="name", how='inner')
df

,title,year,genre,actors,name,nominations
0,Kill Bill: The Whole Bloody Affair,2004,accion,uma thurman,uma thurman,1
1,El señor de los anillos: El retorno del rey,2003,accion,viggo mortensen,viggo mortensen,3
2,El señor de los anillos: El retorno del rey,2003,accion,ian mckellen,ian mckellen,2
3,Reservoir Dogs,1992,accion,tim roth,tim roth,1
4,Reservoir Dogs,1992,accion,harvey keitel,harvey keitel,1
...,...,...,...,...,...,...
116,La novia cadáver,2005,terror,helena bonham carter,helena bonham carter,2
117,El corazón del ángel,1987,terror,mickey rourke,mickey rourke,1
118,El corazón del ángel,1987,terror,robert de niro,robert de niro,7
119,El corazón del ángel,1987,terror,charlotte rampling,charlotte rampling,1


In [ ]:
df.to_json("movies_actors.json", orient="records")

# Género con mas nominaciones

In [ ]:
df = pd.read_json("movies_actors.json")

In [ ]:
df.groupby("genre")["nominations"].sum().sort_values(ascending=False)

,nominations
genre,
drama,139
accion,84
terror,77
comedia,63


**Género** con más nominaciones = DRAMA

Estos números concuerdan con el conocimiento genérico sobre los óscar, puesto que en general las películas más premiadas son dramas. En cambio, las comedias y cine de terror suelen tener menos presencia en los óscar.

In [ ]:
# Total nominaciones por película
df = df.groupby(["title", "genre"])["nominations"].sum()

In [ ]:
df = df.reset_index()
df

,title,genre,nominations
0,Al final de la escalera,terror,7
1,American Beauty,comedia,6
2,American History X,drama,3
3,Blade Runner,accion,1
4,Cadena perpetua,drama,6
...,...,...,...
57,Toy Story,comedia,6
58,Toy Story 3,comedia,6
59,Uno de los nuestros,drama,10
60,Up,comedia,3


In [ ]:
df.groupby("genre")["nominations"].mean()

,nominations
genre,
accion,4.666667
comedia,4.200000
drama,8.176471
terror,6.416667


In [ ]:
df["ratio_nominations"] = df.groupby("genre")["nominations"].transform(lambda x: x/x.mean())
df

,title,genre,nominations,ratio_nominations
0,Al final de la escalera,terror,7,1.090909
1,American Beauty,comedia,6,1.428571
2,American History X,drama,3,0.366906
3,Blade Runner,accion,1,0.214286
4,Cadena perpetua,drama,6,0.733813
...,...,...,...,...
57,Toy Story,comedia,6,1.428571
58,Toy Story 3,comedia,6,1.428571
59,Uno de los nuestros,drama,10,1.223022
60,Up,comedia,3,0.714286


In [ ]:
df.nlargest(3, columns="ratio_nominations")

,title,genre,nominations,ratio_nominations
33,Infiltrados,accion,22,4.714286
42,Los intocables de Eliot Ness,accion,9,1.928571
31,Hannah y sus hermanas,comedia,8,1.904762


In [ ]:
df.nsmallest(3, columns="ratio_nominations")

,title,genre,nominations,ratio_nominations
50,Pesadilla antes de Navidad,terror,1,0.155844
3,Blade Runner,accion,1,0.214286
24,En busca del arca perdida,accion,1,0.214286
